In [1]:
# ====================================================================
# DATA GENERATION FOR CLASSIFICATION AND REGRESSION TASKS
# ====================================================================
# This notebook generates synthetic thermal decomposition curves
# for different kinetic models using ODE solvers
# ====================================================================

import numpy as np
from scipy.integrate import solve_ivp
from tqdm import tqdm

In [ ]:
# ====================================================================
# CONFIGURATION: Data Generation Parameters
# ====================================================================
# These constants control the size and structure of the generated dataset

EPOCHS = 2  # Number of training epochs
BATCH_SIZE = 300  # Samples per batch
STEPS_PER_EPOCH = 15  # Number of batches per epoch (total train = BATCH_SIZE * STEPS_PER_EPOCH)
SEGMENTS = 50  # Number of model combination segments to generate
VALIDATION_STEPS = 2  # Number of batches for validation data
TEST_STEPS = 4  # Number of batches for test data

print("Dataset Configuration:")
print(f"  Training samples per model pair:   {BATCH_SIZE * STEPS_PER_EPOCH:,}")
print(f"  Validation samples per model pair: {BATCH_SIZE * VALIDATION_STEPS:,}")
print(f"  Test samples per model pair:       {BATCH_SIZE * TEST_STEPS:,}")

Dataset Configuration:
  Training samples per model pair:   4,500
  Validation samples per model pair: 600
  Test samples per model pair:       1,200


In [ ]:
# ====================================================================
# PHYSICAL PARAMETERS AND MODEL DEFINITIONS
# ====================================================================
# Defines kinetic models and their parameter ranges for synthetic data generation

from math import log, exp

p = 900  # Input dimension: length of thermogravimetric (TG) curves
q = 5  # Output dimension: number of regression parameters
       # Order: [n_right, n_left, mixing_ratio, logA_right, logA_left]

params = {
    "Ea": 150000,  # Activation Energy (J/mol) - Fixed across all models
    "logA_range": (log(10**10), log(10**15)),  # Pre-exponential factor range for most models
    "logA_Fn_range": (log(10**10), log(10**15)),  # Separate range for Fn model (if different)
    "n_range": (1.0, 3.5),  # Reaction order parameter range (Fn, JMA only)
    "mix_range": (0.1, 0.9),  # Mixing ratio between two kinetic curves
    "rate": 1,  # Heating rate (K/min) - Fixed
    "models": ['D2', 'D3', 'D4', 'R2', 'R3', 'Fn', 'JMA', 'Fn', 'JMA', 'Fn', 'JMA']
}

# Number of UNIQUE kinetic model types (7 total)
# Note: 'models' list has repeats to balance dataset between n-using and non-n-using models
# Fn and JMA (which use n parameter) appear 3x each; others appear once
r = 7

In [ ]:
# ====================================================================
# ODE SOLVER: Sestak-Berggren Kinetic Model (LSODA with Safeguards)
# ====================================================================
# Generates synthetic thermogravimetric (TG) curves by solving ODEs
# for kinetic decomposition models. Uses LSODA per coauthor requirement,
# with stability safeguards to prevent multi-hour stalls observed on Kaggle.
# ====================================================================

def oneSpec(Ea=170000, npoints=4800, A=np.exp(36), m=1, n=0, qqq=10, T0=0,
            TempEnd=500, timeStart=0, y0=5.00082970E-5, model="JMA"):
    """
    Solves ODE for thermal decomposition using Sestak-Berggren kinetic models.

    Stability improvements while keeping LSODA:
    - Safe log wrappers: log_safe(x) uses max(x, eps) to avoid log(0) / inf
    - Denominator guards: max(val, eps) to prevent division overflow
    - Early termination event: stops integration when conversion ~ 0.9999
    - Tolerances (rtol/atol) and max_step to avoid runaway steps
    - Clamp y into [0,1] during rate eval to avoid non-physical drift
    - Fallback: If solver fails or produces NaNs, returns a flat curve

    Parameters:
        Ea (float): Activation Energy (J/mol)
        npoints (int): Number of points in output curve
        A (float): Pre-exponential factor
        m (float): Reaction order parameter (many models)
        n (float): Alternative order parameter (Fn, JMA)
        qqq (float): Heating rate (K/min)
        T0 (float): Initial temperature (°C)
        TempEnd (float): Final temperature (°C)
        timeStart (float): Start time (min)
        y0 (float): Initial conversion
        model (str): Kinetic model name

    Returns:
        dadT (ndarray): Mass loss rate vs temperature curve (length npoints)
    """
    import math

    # --- Setup time and temperature scales ---
    Ts = 273.15 + T0  # Kelvin start temp
    timeEnd = (TempEnd - T0) / (qqq / 60)  # minutes total
    t_eval = np.linspace(start=timeStart, stop=timeEnd, num=npoints)

    R = 8.314  # Gas constant
    eps = 1e-12  # Numerical floor

    def log_safe(x: float) -> float:
        x_clamped = x if x > eps else eps
        return math.log(x_clamped)

    # Early termination event: stop when y ~ 0.9999 (reaction complete)
    def event_complete(t, y, qqq_arg, Ts_arg, Ea_arg, m_arg, n_arg, A_arg, model):
        return y[0] - 0.9999
    event_complete.terminal = True
    event_complete.direction = 1

    def SB_ode(t, y, qqq_arg, Ts_arg, Ea_arg, m_arg, n_arg, A_arg, model):
        y1 = y[0]
        # Clamp y into physical bounds
        if y1 <= 0.0:
            y1 = 0.0
        if y1 >= 1.0:
            return [0.0]
        if math.isnan(y1):
            return [0.0]

        Temperature_current = Ts_arg + t * qqq_arg / 60.0
        coeff = math.exp(-Ea_arg / (R * Temperature_current))

        # Guarded expressions reused
        one_minus_y = (1 - y1) if (1 - y1) > eps else eps
        y_pos = y1 if y1 > eps else eps
        m_pos = m_arg if m_arg > eps else eps
        n_pos = n_arg if n_arg > eps else eps

        # Models
        if model == "RO1":
            dydt = A_arg * coeff * (one_minus_y ** m_arg) * (1 + (n_arg * y1))
        elif model == "RO2":
            # (-log(1 - y))^(1 - 1/n)
            base = -log_safe(one_minus_y)
            dydt = A_arg * coeff * n_arg * one_minus_y * (base ** (1 - (1 / n_pos)))
        elif model == "RO3":
            dydt = A_arg * coeff * (one_minus_y ** m_arg)
        elif model == "SB":
            dydt = A_arg * coeff * (y1 ** n_arg) * (one_minus_y ** m_arg)
        elif model == "P1":
            dydt = A_arg * coeff * 4 * (y_pos ** (3.0 / 4.0))
        elif model == "P2":
            dydt = A_arg * coeff * 3 * (y_pos ** (2.0 / 3.0))
        elif model == "P3":
            dydt = A_arg * coeff * 2 * (y_pos ** 0.5)
        elif model == "P4":
            dydt = A_arg * coeff * (2.0 / 3.0) * (y_pos ** -0.5)
        elif model == "D1":
            dydt = A_arg * coeff * 0.5 / y_pos
        elif model == "An":
            base = -log_safe(one_minus_y)
            dydt = A_arg * coeff * m_arg * one_minus_y * (base ** ((m_arg - 1.0) / m_pos))
        elif model == "D3":
            num = (3.0 / 2.0) * (one_minus_y ** (2.0 / 3.0))
            denom = 1.0 - (one_minus_y ** (1.0 / 3.0))
            denom = denom if denom > eps else eps
            dydt = A_arg * coeff * num / denom
        elif model == "D4":
            # 3/2 / (( (1-y)^(-1/3) ) - 1)
            term = (one_minus_y ** (-1.0 / 3.0)) - 1.0
            term = term if term > eps else eps
            dydt = A_arg * coeff * (3.0 / 2.0) / term
        elif model == "R3":
            dydt = A_arg * coeff * 3.0 * (one_minus_y ** (2.0 / 3.0))
        elif model == "R2":
            dydt = A_arg * coeff * 2.0 * (one_minus_y ** 0.5)
        elif model == "D2":
            # -1 / log(1 - y)
            denom = log_safe(one_minus_y)  # denom negative; guard magnitude
            dydt = A_arg * coeff * -1.0 / denom
        elif model == "JMA":
            base = -log_safe(one_minus_y)
            dydt = A_arg * coeff * m_arg * one_minus_y * (base ** (1.0 - (1.0 / m_pos)))
        elif model == "Ih":
            dydt = A_arg * coeff * y1 * one_minus_y
        elif model == "Fn":
            dydt = A_arg * coeff * (one_minus_y ** m_arg)
        else:
            # Unknown model fallback
            dydt = 0.0
        # Final safety clamp
        if not math.isfinite(dydt):
            dydt = 0.0
        return [dydt]

    try:
        sol = solve_ivp(
            fun=SB_ode,
            t_span=[timeStart, timeEnd],
            y0=[y0],
            method='LSODA',  # Required solver
            t_eval=t_eval,
            args=(qqq, Ts, Ea, m, n, A, model),
            rtol=1e-6,
            atol=1e-9,
            max_step=(timeEnd - timeStart) / npoints * 20,  # limit step growth
            events=event_complete
        )
        yt = sol.y[0]
    except Exception:
        # Fallback: return a zero curve if integration fails
        yt = np.zeros_like(t_eval)

    # --- Normalize output length and enforce physical completion ---
    # Clamp to [0,1]
    yt_raw = np.clip(yt, 0.0, 1.0)

    # Prepare full-length conversion array (handles early termination)
    yt = np.empty(npoints, dtype=float)
    if yt_raw.size == 0:
        yt[:] = 0.0
    else:
        used = min(npoints, yt_raw.size)
        yt[:used] = yt_raw[:used]
        fill_val = float(yt_raw[-1])
        if used < npoints:
            yt[used:] = fill_val

    # Temperature array
    Temperature = Ts + t_eval * qqq / 60.0
    TempStep = Temperature[1] - Temperature[0] if len(Temperature) > 1 else 1.0

    # Ensure monotonic approach to 1.0 after completion
    yt[yt > 1.0] = 1.0
    # Find index where conversion essentially complete
    complete_mask = yt >= 0.9999
    if np.any(complete_mask):
        first_complete = int(np.argmax(complete_mask))
        yt[first_complete:] = 1.0

    # Convert conversion to mass loss percentage (0-100%)
    yt *= 100.0

    return [yt, Temperature, TempStep]

In [ ]:
# ====================================================================
# DATA GENERATION: Synthetic Dataset Creation
# ====================================================================
# Generates training data by mixing two kinetic model curves

def generate_full_dataset(num_samples, npoints, label_dim, params12):
    """
    Generates a full dataset of features (TG curves) and labels (parameters).
    
    Implements curve mixing strategy: creates synthetic samples by weighted
    combination of two kinetic model curves, then normalizes labels for
    neural network training.
    
    Parameters:
    -----------
    num_samples : int
        Number of samples to generate
    npoints : int
        Length of each TG curve (typically 900 or 4800)
    label_dim : int
        Dimensionality of parameter labels (typically 5)
    params12 : dict
        Configuration dict with model pair and parameter ranges:
        - "models": [model1, model2] (str names)
        - "Ea": Activation energy (J/mol)
        - "logA_range": (min, max) for pre-exponential factor
        - "logA_Fn_range": (min, max) for Fn model (may differ)
        - "n_range": (min, max) for reaction order
        - "mix_range": (min, max) for mixing ratio
        - "rate": Heating rate (K/min)
    
    Returns:
    --------
    X : ndarray, shape (num_samples, npoints)
        Feature matrix: normalized TG curves
    y : ndarray, shape (num_samples, label_dim)
        Label matrix: normalized parameters [n_right, n_left, ratio, logA_right, logA_left]
    mins : ndarray
        Minimum values used for normalization (for denormalization)
    ranges : ndarray
        Range (max - min) used for normalization (for denormalization)
    """
    X = np.zeros((num_samples, npoints, 3), dtype=np.float32)
    y = np.zeros((num_samples, label_dim), dtype=np.float32)

    # --- Extract parameter ranges from configuration ---
    Ea = params12["Ea"]
    logA1_min, logA1_max = params12["logA_range"]
    logA2_min, logA2_max = params12["logA_range"]
    logA_Fn1_min, logA_Fn1_max = params12["logA_Fn_range"]
    logA_Fn2_min, logA_Fn2_max = params12["logA_Fn_range"]
    m1_min, m1_max = params12["n_range"]
    m2_min, m2_max = params12["n_range"]
    mix_min, mix_max = params12["mix_range"]
    qqq = params12["rate"]
    models = params12["models"]

    # --- Define normalization bounds ---
    # Label order: [m2, m1, mix, logA2, logA1]
    mins = np.array([[m2_min, m1_min, mix_min, logA2_min, logA1_min]], dtype=np.float32)
    maxs = np.array([[m2_max, m1_max, mix_max, logA2_max, logA1_max]], dtype=np.float32)
    
    # Adjust ranges for Fn model if needed
    if models[0] == "Fn":
        mins[0, 4] = logA_Fn1_min
        maxs[0, 4] = logA_Fn1_max
    if models[1] == "Fn":
        mins[0, 3] = logA_Fn2_min
        maxs[0, 3] = logA_Fn2_max

    ranges = maxs - mins

    # --- Generate samples with progress tracking ---
    for i in range(num_samples):
        # Sample parameters for first curve
        if models[0] == "Fn":
            rand_logA1 = np.random.uniform(logA_Fn1_min, logA_Fn1_max)
        else:
            rand_logA1 = np.random.uniform(logA1_min, logA1_max)
        
        # Sample parameters for second curve
        if models[1] == "Fn":
            rand_logA2 = np.random.uniform(logA_Fn2_min, logA_Fn2_max)
        else:
            rand_logA2 = np.random.uniform(logA2_min, logA2_max)
        
        # Ensure logA1 < logA2 (ordering constraint)
        if rand_logA1 > rand_logA2:
            tmp = rand_logA1
            rand_logA1 = rand_logA2
            rand_logA2 = tmp
        
        # Sample reaction order and mixing parameters
        rand_m1 = np.random.uniform(m1_min, m1_max)
        rand_m2 = np.random.uniform(m2_min, m2_max)
        rand_mix = np.random.uniform(mix_min, mix_max)

        # Generate TG curve for first model
        [y1, temp1, TempStep1] = oneSpec(Ea=Ea, npoints=npoints, A=np.exp(rand_logA1), m=rand_m1,
                        qqq=qqq, T0=0, TempEnd=500, timeStart=0,
                        y0=5.00082970E-5, model=models[0])
        [y2, _, _] = oneSpec(Ea=Ea, npoints=npoints, A=np.exp(rand_logA2), m=rand_m2,
                        qqq=qqq, T0=0, TempEnd=500, timeStart=0,
                        y0=5.00082970E-5, model=models[1])

        # Mix curves
        combined_y = rand_mix * y1 + (1 - rand_mix) * y2
        # Compute derivative wrt Temperature
        combined_dadT = np.diff(combined_y, prepend=combined_y[0]) / TempStep1
        # Second derivative
        combined_d2adT2 = np.diff(combined_dadT, prepend=combined_dadT[0]) / TempStep1
        # Derivative scaled by squared temperature
        combined_dadT_scaled = combined_dadT * (temp1 ** 2)

        # Store feature and label
        X[i, :, 0] = combined_dadT
        X[i, :, 1] = combined_d2adT2
        X[i, :, 2] = combined_dadT_scaled
        y[i, :] = [rand_m2, rand_m1, rand_mix, rand_logA2, rand_logA1]
    
    # --- Normalize labels to [0, 1] range for neural network ---
    y -= mins
    y /= ranges

    return X, y, mins, ranges

In [ ]:
# ====================================================================
# GENERATE FULL DATASET: All Model Combinations
# ====================================================================
# Creates clean TG spectra for training, validation, and test sets
# Derivative features are created downstream after optional test noise

import gc

# Calculate total samples needed for each split
total_train_samples = BATCH_SIZE * STEPS_PER_EPOCH
total_val_samples = BATCH_SIZE * VALIDATION_STEPS
total_test_samples = BATCH_SIZE * TEST_STEPS

# Calculate number of model pairs
num_model_pairs = len(params["models"]) ** 2
total_pair_count = SEGMENTS * num_model_pairs

# Calculate final dataset sizes
final_train_size = total_train_samples * num_model_pairs
final_val_size = total_val_samples * num_model_pairs
final_test_size = total_test_samples * num_model_pairs

# Pre-allocate clean TG spectra and labels
X_train = np.zeros((final_train_size, p), dtype=np.float32)
y_train = np.zeros((final_train_size, q), dtype=np.float32)
X_val = np.zeros((final_val_size, p), dtype=np.float32)
y_val = np.zeros((final_val_size, q), dtype=np.float32)
X_test = np.zeros((final_test_size, p), dtype=np.float32)
y_test = np.zeros((final_test_size, q), dtype=np.float32)

# Temperature metadata is shared by all generated spectra
temperature = np.linspace(273.15, 773.15, p, dtype=np.float32)
temp_step = float(temperature[1] - temperature[0])

# Pre-allocate label arrays using fixed-width unicode
labels_train = np.empty((final_train_size, 2), dtype="<U6")
labels_val = np.empty((final_val_size, 2), dtype="<U6")
labels_test = np.empty((final_test_size, 2), dtype="<U6")

# Print generation summary
print(f"Generating {final_train_size} training samples...")
print(f"Generating {final_val_size} validation samples...")
print(f"Generating {final_test_size} test samples...")
print(f"Using {total_pair_count} model pair combinations ({SEGMENTS} segments x {num_model_pairs} pairs)\n")

# Track current index in each pre-allocated array
train_idx = 0
val_idx = 0
test_idx = 0

# Compute per-segment base counts and remainders (per pair)
train_base = total_train_samples // SEGMENTS
train_rem = total_train_samples % SEGMENTS
val_base = total_val_samples // SEGMENTS
val_rem = total_val_samples % SEGMENTS
test_base = total_test_samples // SEGMENTS
test_rem = total_test_samples % SEGMENTS

# Double loop: generate all combinations of model pairs across segments
for segment in tqdm(range(SEGMENTS), desc="Segments"):
    seg_train_count = train_base + (1 if segment < train_rem else 0)
    seg_val_count = val_base + (1 if segment < val_rem else 0)
    seg_test_count = test_base + (1 if segment < test_rem else 0)

    for model1 in params["models"]:
        for model2 in params["models"]:
            params12 = params.copy()
            params12["models"] = [model1, model2]

            # --- TRAIN ---
            if seg_train_count > 0:
                X_train12, y_train12, mins, ranges = generate_full_dataset(
                    seg_train_count, p, q, params12
                )
                X_train[train_idx:train_idx + seg_train_count] = X_train12
                y_train[train_idx:train_idx + seg_train_count] = y_train12
                labels_train[train_idx:train_idx + seg_train_count, 0] = model1
                labels_train[train_idx:train_idx + seg_train_count, 1] = model2
                train_idx += seg_train_count
                del X_train12, y_train12

            # --- VALIDATION ---
            if seg_val_count > 0:
                X_val12, y_val12, _, _ = generate_full_dataset(
                    seg_val_count, p, q, params12
                )
                X_val[val_idx:val_idx + seg_val_count] = X_val12
                y_val[val_idx:val_idx + seg_val_count] = y_val12
                labels_val[val_idx:val_idx + seg_val_count, 0] = model1
                labels_val[val_idx:val_idx + seg_val_count, 1] = model2
                val_idx += seg_val_count
                del X_val12, y_val12

            # --- TEST ---
            if seg_test_count > 0:
                X_test12, y_test12, _, _ = generate_full_dataset(
                    seg_test_count, p, q, params12
                )
                X_test[test_idx:test_idx + seg_test_count] = X_test12
                y_test[test_idx:test_idx + seg_test_count] = y_test12
                labels_test[test_idx:test_idx + seg_test_count, 0] = model1
                labels_test[test_idx:test_idx + seg_test_count, 1] = model2
                test_idx += seg_test_count
                del X_test12, y_test12

            # Periodic garbage collection to free intermediate memory
            if (segment * num_model_pairs + params["models"].index(model1) * len(params["models"]) + params["models"].index(model2)) % 25 == 0:
                gc.collect()

print("\n✓ All model pairs generated and filled into pre-allocated arrays")

# Integrity checks
assert train_idx == final_train_size, f"Train index mismatch: {train_idx} != {final_train_size}"
assert val_idx == final_val_size, f"Val index mismatch: {val_idx} != {final_val_size}"
assert test_idx == final_test_size, f"Test index mismatch: {test_idx} != {final_test_size}"

# Quick diagnostics on label fill completeness
filled_train = np.count_nonzero(labels_train[:, 0] != '')
filled_val = np.count_nonzero(labels_val[:, 0] != '')
filled_test = np.count_nonzero(labels_test[:, 0] != '')
print(f"Filled labels -> train: {filled_train}/{final_train_size}, val: {filled_val}/{final_val_size}, test: {filled_test}/{final_test_size}")

unique_left = np.unique(labels_train[:, 0])
unique_right = np.unique(labels_train[:, 1])
print(f"Left label unique count: {len(unique_left)} -> {unique_left}")
print(f"Right label unique count: {len(unique_right)} -> {unique_right}")

# Ensure no empty or stray values
assert '' not in unique_left and 'None' not in unique_left, "Left labels contain empty/None"
assert '' not in unique_right and 'None' not in unique_right, "Right labels contain empty/None"

Generating 544500 training samples...
Generating 72600 validation samples...
Generating 145200 test samples...
Using 6050 model pair combinations (50 segments × 121 pairs)



Segments: 100%|██████████| 50/50 [4:01:58<00:00, 290.37s/it]



✓ All model pairs generated and filled into pre-allocated arrays
Filled labels -> train: 544500/544500, val: 72600/72600, test: 145200/145200
Left label unique count: 7 -> ['D2' 'D3' 'D4' 'Fn' 'JMA' 'R2' 'R3']
Right label unique count: 7 -> ['D2' 'D3' 'D4' 'Fn' 'JMA' 'R2' 'R3']


In [7]:
# ====================================================================
# ENCODE MODEL LABELS TO INTEGER CLASSES
# ====================================================================
# Convert model names (strings) to integer class indices using LabelEncoder

from sklearn.preprocessing import LabelEncoder
import gc

le = LabelEncoder()
# Fit encoder on unique model names
y_left_train = le.fit_transform(labels_train[:, 0].ravel())
y_right_train = le.transform(labels_train[:, 1].ravel())
y_left_val = le.transform(labels_val[:, 0].ravel())
y_right_val = le.transform(labels_val[:, 1].ravel())
y_left_test = le.transform(labels_test[:, 0].ravel())
y_right_test = le.transform(labels_test[:, 1].ravel())

print(f"Model class mapping:")
for idx, name in enumerate(le.classes_):
    print(f"  {idx} → {name}")

# ====================================================================
# CLEANUP: Delete intermediate label arrays after encoding
# ====================================================================
# Labels were only needed for encoding; now we have integer class labels
print("\nCleaning up intermediate label arrays...")
del labels_train, labels_val, labels_test
gc.collect()
print("✓ Freed intermediate label arrays")

Model class mapping:
  0 → D2
  1 → D3
  2 → D4
  3 → Fn
  4 → JMA
  5 → R2
  6 → R3

Cleaning up intermediate label arrays...
✓ Freed intermediate label arrays


In [ ]:
# ====================================================================
# PERSIST GENERATED DATASET TO DISK
# ====================================================================
# Save clean TG spectra and parameters for downstream notebooks

import pickle

output_data = {
    "X_train": X_train,
    "y_train": y_train,
    "y_left_train": y_left_train,
    "y_right_train": y_right_train,
    "X_val": X_val,
    "y_val": y_val,
    "y_left_val": y_left_val,
    "y_right_val": y_right_val,
    "X_test": X_test,
    "y_test": y_test,
    "y_left_test": y_left_test,
    "y_right_test": y_right_test,
    "temperature": temperature,
    "temp_step": temp_step,
    "mins": mins,
    "ranges": ranges,
    "params": params,
    "p": p,
    "q": q,
    "r": r,
    "BATCH_SIZE": BATCH_SIZE,
    "STEPS_PER_EPOCH": STEPS_PER_EPOCH,
    "VALIDATION_STEPS": VALIDATION_STEPS,
    "TEST_STEPS": TEST_STEPS,
    "SEGMENTS": SEGMENTS,
    "EPOCHS": EPOCHS
}

with open("/kaggle/working/data_classification.pkl", "wb") as f:
    pickle.dump(output_data, f)

print("✓ Dataset saved to disk")

✓ Dataset saved to disk
